In [ ]:
import time
import statistics     # Gets tools to calculate averages and other math operations
import pickle         # save data to files and load them back later
import os             # Helps check if files exist on your computer
import numpy as np    # Powerful math library for working with numbers and arrays
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

class WutheringWavesTypingBiometrics:
    def __init__(self):     #This runs automatically when we create the system
        self.target_sentence = "The Wuthering waves ebb and flow.They shall send you back to where you belong."
        self.scaler = StandardScaler() # A tool that makes numbers more uniform for better machine learning
        self.model = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=10) #"brain" that learns your typing pattern

     #################################################
          # class - Creates a blueprint for our typing system
     #################################################
    def capture_typing_biometrics(self):  #Defines a function (a set of instructions)
        """Capture typing biometrics for the specific sentence"""
        print(f"\nPlease type the following sentence exactly:")
        print(f"'{self.target_sentence}'")
        print("\nPress Enter when ready to start...")
        input("Ready? Press Enter...")
        
        print(f"\nNow type: {self.target_sentence}")
        
        # Capture typing
        start_time = time.time()
        typed_text = input("> ").strip()
        end_time = time.time()
        
        # Calculate biometric features
        features = self.analyze_typing_pattern(typed_text, start_time, end_time)
        return features
    
    def analyze_typing_pattern(self, typed_text, start_time, end_time):
        """Analyze the typing pattern and extract biometric features"""
        total_time = end_time - start_time
        total_chars = len(typed_text)
        
        # Basic timing features
        typing_speed = total_chars / total_time if total_time > 0 else 0
        
        # Word-level analysis
        typed_words = typed_text.split()
        target_words = self.target_sentence.split()
        
        words_per_minute = (len(typed_words) * 60) / total_time if total_time > 0 else 0
        
        # Accuracy calculations
        character_accuracy = self.calculate_character_accuracy(self.target_sentence, typed_text)
        word_accuracy = self.calculate_word_accuracy(target_words, typed_words)
        
        # Rhythm analysis
        avg_time_per_word = total_time / len(typed_words) if typed_words else 0
        avg_time_per_char = total_time / total_chars if total_chars > 0 else 0
        
        features = {
            'total_time': total_time,
            'typing_speed': typing_speed,
            'words_per_minute': words_per_minute,
            'character_accuracy': character_accuracy,
            'word_accuracy': word_accuracy,
            'avg_time_per_word': avg_time_per_word,
            'avg_time_per_char': avg_time_per_char,
            'total_chars': total_chars,
            'total_words': len(typed_words)
        }
        
        return features
    
    def features_to_vector(self, features):
        """Convert features dictionary to numpy array for ML model"""
        return np.array([
            features['total_time'],
            features['typing_speed'],
            features['words_per_minute'],
            features['character_accuracy'],
            features['word_accuracy'],
            features['avg_time_per_word'],
            features['avg_time_per_char'],
            features['total_chars'],
            features['total_words']
        ])
                        ########## Compares what you typed character by character with the target sentence  ########
    def calculate_character_accuracy(self, expected, typed):
        """Calculate character-level accuracy"""
        if len(expected) == 0:
            return 0.0
        
        min_length = min(len(expected), len(typed))
        correct_chars = sum(1 for i in range(min_length) if expected[i] == typed[i])
        
        # Penalize for length differences
        length_penalty = abs(len(expected) - len(typed)) / len(expected)
        accuracy = (correct_chars / len(expected)) - length_penalty
        
        return max(0.0, accuracy)
    
    def calculate_word_accuracy(self, expected_words, typed_words):
        """Calculate word-level accuracy"""
        if len(expected_words) == 0:
            return 0.0
        
        correct_words = 0
        min_words = min(len(expected_words), len(typed_words))
        
        for i in range(min_words):
            if expected_words[i].lower() == typed_words[i].lower():
                correct_words += 1
        
        return correct_words / len(expected_words)
        
        ######## Registration process

    def register_user(self, user_id, num_datasets=5):   #### Repeats 5 times (collects 5 typing samples)
        """Register user using machine learning approach"""
        print(f"\n=== Registration for {user_id} ===")
        print(f"Target Sentence: '{self.target_sentence}'")
        print(f"Creating {num_datasets} datasets for ML training")
        
        genuine_samples = []
        successful_datasets = 0
        
        # Collect genuine user samples
        for dataset_num in range(num_datasets):
            print(f"\n--- Dataset {dataset_num + 1}/{num_datasets} ---")
            
            features = self.capture_typing_biometrics()
            
            if features and features['character_accuracy'] > 0.7:   ### Only accepts samples with 70% accuracy or higher
                feature_vector = self.features_to_vector(features)
                genuine_samples.append(feature_vector)
                successful_datasets += 1
                
                print(f"Dataset {dataset_num + 1} recorded!")
                print(f"   Speed: {features['typing_speed']:.1f} chars/sec")
                print(f"   Accuracy: {features['character_accuracy']:.1%}")
                print(f"   WPM: {features['words_per_minute']:.1f}")
            else:
                print(f"Dataset {dataset_num + 1} failed - accuracy too low")
        
        if successful_datasets < 3:
            print("Registration failed - need at least 3 successful datasets")
            return False
        
        # Generate synthetic impostor samples for training
        print("Generating synthetic impostor samples...")
        impostor_samples = []
        
        for genuine_sample in genuine_samples:
            # Create 3 impostor variants per genuine sample
            for _ in range(3):
                impostor_sample = []
                for feature in genuine_sample:
                    if feature > 0:
                        # Add noise to simulate impostor typing
                        noise_factor = np.random.uniform(0.3, 0.8)
                        noise = np.random.normal(0, feature * noise_factor)
                        impostor_sample.append(max(0.01, feature + noise))
                    else:
                        impostor_sample.append(0)
                
                impostor_samples.append(np.array(impostor_sample))
        
        # Prepare training data
        X = np.vstack([genuine_samples, impostor_samples])  #All the typing patterns (yours and fake impostors)
        y = np.hstack([np.ones(len(genuine_samples)), np.zeros(len(impostor_samples))])   # Labels: 1 for "real you", 0 for "impostor"
        
        # Split data for training and validation
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
        
        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        # Train ML model
        print("Training Random Forest model...")
        self.model.fit(X_train_scaled, y_train)
        
        # Evaluate model
        train_accuracy = self.model.score(X_train_scaled, y_train)
        test_accuracy = self.model.score(X_test_scaled, y_test)
        
        print(f"Model Training Accuracy: {train_accuracy:.3f}")
        print(f"Model Test Accuracy: {test_accuracy:.3f}")
        
        # Save model and scaler
        model_data = {
            'model': self.model,
            'scaler': self.scaler,
            'user_id': user_id,
            'sentence': self.target_sentence,
            'training_samples': successful_datasets,
            'genuine_samples': genuine_samples,
            'model_accuracy': test_accuracy
        }
        
        filename = f"{user_id}_ml_model.pkl"
        with open(filename, 'wb') as f:
            pickle.dump(model_data, f)
        
        print(f"\nRegistration Complete!")
        print(f"User: {user_id}")
        print(f"Training Datasets: {successful_datasets}")
        print(f"Model Accuracy: {test_accuracy:.1%}")
        print(f"Model saved: {filename}")
        
        return True
    
    def authenticate_user(self, user_id):
        """Authenticate user using trained ML model"""
        filename = f"{user_id}_ml_model.pkl"
        
        if not os.path.exists(filename):
            print(f"No model found for user: {user_id}")
            print("Please register first")
            return False, 0.0
        
        # Load trained model
        with open(filename, 'rb') as f:
            model_data = pickle.load(f)
            trained_model = model_data['model']
            scaler = model_data['scaler']
            training_samples = model_data['training_samples']
            model_accuracy = model_data['model_accuracy']
        
        print(f"\n=== Authentication for {user_id} ===")
        print(f"Model trained on {training_samples} datasets")
        print(f"Model accuracy: {model_accuracy:.1%}")
        
        # Capture test sample
        test_features = self.capture_typing_biometrics()
        
        if not test_features or test_features['character_accuracy'] < 0.5:
            print("Authentication failed - typing accuracy too low")
            return False, 0.0
        
        # Convert to feature vector and scale
        test_vector = self.features_to_vector(test_features).reshape(1, -1)
        test_vector_scaled = scaler.transform(test_vector)
        
        # Make prediction
        prediction = trained_model.predict(test_vector_scaled)[0]
        prediction_proba = trained_model.predict_proba(test_vector_scaled)[0]
        
        # Get confidence scores
        genuine_confidence = prediction_proba[1]  # Probability of being genuine user
        impostor_confidence = prediction_proba[0]  # Probability of being impostor
        
        # Authentication decision
        threshold = 0.7
        is_authenticated = prediction == 1 and genuine_confidence >= threshold
        
        print(f"\nAuthentication Analysis:")
        print(f"Test Speed: {test_features['typing_speed']:.1f} chars/sec")
        print(f"Test WPM: {test_features['words_per_minute']:.1f}")
        print(f"Test Time: {test_features['total_time']:.2f}s")
        print(f"Test Accuracy: {test_features['character_accuracy']:.1%}")
        print()
        print(f"ML Prediction: {'Genuine User' if prediction == 1 else 'Impostor'}")
        print(f"Genuine Confidence: {genuine_confidence:.3f}")
        print(f"Impostor Risk: {impostor_confidence:.3f}")
        print(f"Authentication Threshold: {threshold}")
        
        if is_authenticated:
            print(f"\nAUTHENTICATION SUCCESSFUL")
            print(f"Welcome back, {user_id}!")
            print(f"ML Confidence: {genuine_confidence*100:.1f}%")
        else:
            print(f"\nAUTHENTICATION FAILED")
            if prediction == 0:
                print("Model classified as impostor")
            else:
                print(f"Confidence below threshold ({genuine_confidence:.3f} < {threshold})")
        
        return is_authenticated, genuine_confidence

def main_menu():
    """Main menu system"""
    system = WutheringWavesTypingBiometrics()
    
    while True:
        print("\n" + "=" * 50)
        print("  WUTHERING WAVES TYPING BIOMETRICS SYSTEM")
        print("=" * 50)
        print()
        print("1. Register user")
        print("2. Authenticate user")
        print("3. Exit")
        
        choice = input("\nSelect option: ").strip()  # Removes extra spaces from what you typed
        
        if choice == "1":
            user_id = input("Enter username: ").strip()
            if user_id:
                datasets = input("Number of datasets (3-8) [default: 5]: ").strip() # Removes extra spaces from what you typed+
                datasets = int(datasets) if datasets.isdigit() else 5
                datasets = max(3, min(8, datasets))
                system.register_user(user_id, datasets)
            else:
                print("Please enter a valid username")
                
        elif choice == "2":
            user_id = input("Enter username: ").strip()
            if user_id:
                success, confidence = system.authenticate_user(user_id)
            else:
                print("Please enter a valid username")
            
        elif choice == "3":
            print("\nThank you Rover for using Wuthering Typing Biometrics!")
            print("Goodbye!")
            break
            
        else:
            print("Invalid option. Please select 1, 2, or 3.")

# Initialize and run the system
print("Wuthering Waves Typing Biometrics System")
print("This system uses machine learning for typing pattern authentication.")

# Start the main menu
main_menu()


Wuthering Waves Typing Biometrics System
This system uses machine learning for typing pattern authentication.

  WUTHERING WAVES TYPING BIOMETRICS SYSTEM

1. Register user
2. Authenticate user
3. Exit



Select option:  1
Enter username:  Arush04
Number of datasets (3-8) [default: 5]:  3



=== Registration for Arush04 ===
Target Sentence: 'The Wuthering waves ebb and flow.They shall send you back to where you belong.'
Creating 3 datasets for ML training

--- Dataset 1/3 ---

Please type the following sentence exactly:
'The Wuthering waves ebb and flow.They shall send you back to where you belong.'

Press Enter when ready to start...


Ready? Press Enter... 



Now type: The Wuthering waves ebb and flow.They shall send you back to where you belong.


...


=== Authentication for Arush04 ===
Model trained on 3 datasets
Model accuracy: 75.0%

Please type the following sentence exactly:
'The Wuthering waves ebb and flow.They shall send you back to where you belong.'

Press Enter when ready to start...


Ready? Press Enter... 



Now type: The Wuthering waves ebb and flow.They shall send you back to where you belong.


>  The Wuthering waves ebb and flow.They shall send you back to where you belong.



Authentication Analysis:
Test Speed: 2.9 chars/sec
Test WPM: 30.7
Test Time: 27.33s
Test Accuracy: 100.0%

ML Prediction: Genuine User
Genuine Confidence: 0.940
Impostor Risk: 0.060
Authentication Threshold: 0.7

AUTHENTICATION SUCCESSFUL
Welcome back, Arush04!
ML Confidence: 94.0%

  WUTHERING WAVES TYPING BIOMETRICS SYSTEM

1. Register user
2. Authenticate user
3. Exit
